In [9]:
%%html
<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>Playable Chess Game</title>

    <style>
        body {
            font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif;
            background-color: #222;
            color: #fff;
            display: flex;
            flex-direction: column;
            align-items: center;
            justify-content: center;
            padding: 10px;
            margin: 0;
        }
        h1 {
            margin-bottom: 5px;
            font-size: 1.8rem;
        }
        #hint {
            font-size: 0.85rem;
            color: #aaa;
            margin-bottom: 10px;
        }
        #controls {
            margin-bottom: 15px;
            display: flex;
            gap: 10px;
            align-items: center;
        }
        button, select {
            padding: 6px 12px;
            font-size: 0.9rem;
            background-color: #4CAF50;
            color: white;
            border: none;
            border-radius: 5px;
            cursor: pointer;
        }
        select {
            background-color: #444;
            border: 1px solid #666;
        }
        button:hover {
            background-color: #45a049;
        }

        /* Custom board rendered with plain CSS grid + Unicode glyphs -
           no external image files or third-party board library needed */
        #myBoard {
            display: grid;
            grid-template-columns: repeat(8, 1fr);
            grid-template-rows: repeat(8, 1fr);
            width: 352px;
            height: 352px;
            border: 3px solid #5a3a22;
            box-shadow: 0 4px 12px rgba(0,0,0,0.5);
        }
        .square {
            display: flex;
            align-items: center;
            justify-content: center;
            font-size: 2.1rem;
            line-height: 1;
            cursor: pointer;
            user-select: none;
            position: relative;
        }
        .light { background-color: #f0d9b5; }
        .dark { background-color: #b58863; }
        .selected { outline: 3px solid #f5c518 inset; box-shadow: inset 0 0 0 3px #f5c518; }
        .legal-move::after {
            content: '';
            position: absolute;
            width: 26%;
            height: 26%;
            border-radius: 50%;
            background: rgba(20, 20, 20, 0.35);
        }
        .legal-move.has-piece::after {
            width: 100%;
            height: 100%;
            border-radius: 0;
            background: rgba(200, 40, 40, 0.35);
        }
        .white-piece {
            color: #fdfdfd;
            text-shadow: -1px -1px 0 #222, 1px -1px 0 #222, -1px 1px 0 #222, 1px 1px 0 #222;
        }
        .black-piece {
            color: #1a1a1a;
        }

        #status {
            margin-top: 15px;
            font-size: 1.1rem;
            font-weight: bold;
            text-align: center;
        }

        #celebration-overlay {
            display: none;
            position: fixed;
            top: 0;
            left: 0;
            width: 100vw;
            height: 100vh;
            background: rgba(0, 0, 0, 0.85);
            z-index: 9999;
            flex-direction: column;
            align-items: center;
            justify-content: center;
            text-align: center;
        }
        #celebration-text {
            font-size: 2rem;
            font-weight: bold;
            margin-bottom: 20px;
        }
        #emoji-blast {
            font-size: 4rem;
            margin-bottom: 20px;
        }
    </style>
</head>
<body>

    <h1>Play Against AI</h1>
    <div id="hint">Click a piece, then click a square to move it</div>

    <div id="controls">
        <div>
            <label for="difficulty">AI: </label>
            <select id="difficulty">
                <option value="1">Easy</option>
                <option value="2" selected>Medium</option>
                <option value="3">Hard</option>
            </select>
        </div>
        <button id="resetBtn">Restart</button>
    </div>

    <div id="myBoard"></div>

    <div id="status">Your turn (White)</div>

    <div id="celebration-overlay">
        <div id="emoji-blast">🎉🎈💥🎉💥🎈🎉</div>
        <div id="celebration-text">You Win!</div>
        <button id="overlayCloseBtn">Play Again</button>
    </div>

    <!-- Only dependency left is chess.js (game rules / legal moves).
         The board itself is now plain HTML/CSS/JS - no image assets,
         no third-party board widget, no jQuery. -->
    <script src="https://cdnjs.cloudflare.com/ajax/libs/chess.js/0.10.3/chess.min.js"></script>

    <script>
        let game = new Chess();
        let thinking = false;
        let selectedSquare = null;
        let legalTargets = [];

        const FILES = 'abcdefgh';
        const PIECE_UNICODE = {
            w: { p: '♙', n: '♘', b: '♗', r: '♖', q: '♕', k: '♔' },
            b: { p: '♟', n: '♞', b: '♝', r: '♜', q: '♛', k: '♚' }
        };

        const boardEl = document.getElementById('myBoard');
        const statusEl = document.getElementById('status');
        const overlayEl = document.getElementById('celebration-overlay');
        const overlayTextEl = document.getElementById('celebration-text');

        // --- Self-contained minimax AI (no Worker, no external engine) ---
        const PIECE_VALUES = { p: 100, n: 320, b: 330, r: 500, q: 900, k: 0 };

        function evaluateBoard(g) {
            let total = 0;
            for (let f = 0; f < 8; f++) {
                for (let r = 1; r <= 8; r++) {
                    const piece = g.get(FILES[f] + r);
                    if (piece) {
                        const val = PIECE_VALUES[piece.type];
                        total += piece.color === 'w' ? val : -val;
                    }
                }
            }
            return total;
        }

        function minimax(g, depth, alpha, beta, maximizing) {
            if (depth === 0 || g.game_over()) {
                return evaluateBoard(g);
            }
            const moves = g.moves();
            if (maximizing) {
                let best = -Infinity;
                for (const m of moves) {
                    g.move(m);
                    best = Math.max(best, minimax(g, depth - 1, alpha, beta, false));
                    g.undo();
                    alpha = Math.max(alpha, best);
                    if (beta <= alpha) break;
                }
                return best;
            } else {
                let best = Infinity;
                for (const m of moves) {
                    g.move(m);
                    best = Math.min(best, minimax(g, depth - 1, alpha, beta, true));
                    g.undo();
                    beta = Math.min(beta, best);
                    if (beta <= alpha) break;
                }
                return best;
            }
        }

        function findBestMove(g, depth) {
            const moves = g.moves();
            if (depth === 1 && Math.random() < 0.35) {
                return moves[Math.floor(Math.random() * moves.length)];
            }
            let bestMove = null;
            let bestValue = Infinity;
            for (const m of moves) {
                g.move(m);
                const value = minimax(g, depth - 1, -Infinity, Infinity, true);
                g.undo();
                if (value < bestValue) {
                    bestValue = value;
                    bestMove = m;
                }
            }
            return bestMove || moves[0];
        }

        function makeAIMove() {
            if (game.game_over() || thinking) return;
            thinking = true;
            statusEl.innerHTML = 'AI is thinking...';
            window.setTimeout(() => {
                const depth = parseInt(document.getElementById('difficulty').value, 10);
                const move = findBestMove(game, depth);
                if (move) {
                    game.move(move);
                }
                thinking = false;
                renderBoard();
                updateStatus();
            }, 50);
        }

        function handleSquareClick(square) {
            if (game.game_over() || thinking || game.turn() !== 'w') return;

            if (selectedSquare) {
                if (selectedSquare === square) {
                    selectedSquare = null;
                    legalTargets = [];
                    renderBoard();
                    return;
                }
                const move = game.move({ from: selectedSquare, to: square, promotion: 'q' });
                selectedSquare = null;
                legalTargets = [];
                if (move) {
                    renderBoard();
                    updateStatus();
                    if (!game.game_over()) {
                        window.setTimeout(makeAIMove, 250);
                    }
                    return;
                }
                // not legal - fall through in case they clicked another own piece
            }

            const piece = game.get(square);
            if (piece && piece.color === 'w') {
                selectedSquare = square;
                legalTargets = game.moves({ square: square, verbose: true }).map(m => m.to);
            }
            renderBoard();
        }

        function renderBoard() {
            boardEl.innerHTML = '';
            for (let rank = 8; rank >= 1; rank--) {
                for (let f = 0; f < 8; f++) {
                    const file = FILES[f];
                    const square = file + rank;
                    const isLight = (f + rank) % 2 === 0;

                    const div = document.createElement('div');
                    div.className = 'square ' + (isLight ? 'light' : 'dark');

                    const piece = game.get(square);
                    if (piece) {
                        div.textContent = PIECE_UNICODE[piece.color][piece.type];
                        div.classList.add(piece.color === 'w' ? 'white-piece' : 'black-piece');
                    }
                    if (square === selectedSquare) div.classList.add('selected');
                    if (legalTargets.includes(square)) {
                        div.classList.add('legal-move');
                        if (piece) div.classList.add('has-piece');
                    }

                    div.addEventListener('click', () => handleSquareClick(square));
                    boardEl.appendChild(div);
                }
            }
        }

        function triggerWinScreen(message) {
            overlayTextEl.innerHTML = message;
            overlayEl.style.display = 'flex';
        }

        function updateStatus() {
            let status = '';
            if (game.in_checkmate()) {
                if (game.turn() === 'b') {
                    triggerWinScreen("Victory!<br>You defeated the AI! 🎉");
                    status = "Game Over - You win!";
                } else {
                    triggerWinScreen("Game Over!<br>The AI won. 🤖");
                    status = "Game Over - AI wins.";
                }
            } else if (game.in_draw()) {
                triggerWinScreen("It's a Draw! 🤝");
                status = "Game Over - Draw.";
            } else {
                status = (game.turn() === 'w') ? "Your turn (White)" : "AI is thinking...";
                if (game.in_check()) {
                    status += ' - Check! ⚠️';
                }
            }
            statusEl.innerHTML = status;
        }

        function resetMatch() {
            game.reset();
            thinking = false;
            selectedSquare = null;
            legalTargets = [];
            renderBoard();
            updateStatus();
            overlayEl.style.display = 'none';
        }

        document.getElementById('resetBtn').addEventListener('click', resetMatch);
        document.getElementById('overlayCloseBtn').addEventListener('click', resetMatch);

        renderBoard();
        updateStatus();
    </script>
</body>
</html>